In [1]:
!pip install fastapi uvicorn httpx pyngrok
!pip install transformers accelerate

In [2]:
from pyngrok import ngrok
ngrok.set_auth_token("3Bboq8evmPPjvnGifkVIolo0Y9C_61mcqzGXXZttiF9FbQcv2")

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
REGISTRY_SECRET = user_secrets.get_secret("REGISTRY_SECRET")
REGISTRY_URL = user_secrets.get_secret("REGISTRY_URL")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
import os, uuid, asyncio, threading, time
import torch
import httpx
import torchaudio
import nest_asyncio
from fastapi import FastAPI, File, UploadFile, Header, HTTPException
import uvicorn
from transformers import AutoProcessor, SeamlessM4Tv2Model
import io

# 🔧 allow uvicorn inside notebook
nest_asyncio.apply()

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "seamless-large-M4T"
SESSION_ID = str(uuid.uuid4())

processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large").to("cuda")
app = FastAPI()

@app.post("/transcribe_audio")
async def transcribe_audio(
    audio: UploadFile = File(...),
    x_registry_token: str = Header(None)
):
    if REGISTRY_SECRET and x_registry_token != REGISTRY_SECRET:
        raise HTTPException(status_code=403, detail="Forbidden")
    audio_bytes = await audio.read()
    waveform, sample_rate = torchaudio.load(io.BytesIO(audio_bytes))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sample_rate != 16000:
        waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)
    inputs = processor(audio=waveform, sampling_rate=16000, return_tensors="pt").to("cuda")
    def run_inference():
        with torch.no_grad():
            output_tokens = model.generate(**inputs, tgt_lang="urd", generate_speech=False)
        return processor.decode(output_tokens[0].tolist(), skip_special_tokens=True)
    transcript = await asyncio.to_thread(run_inference)
    return {"transcript": ' '.join(transcript)}

def registry_loop(public_url: str):
    # ✅ NEVER allow None in headers
    headers = {
        k: v for k, v in {
            "X-Registry-Token": REGISTRY_SECRET
        }.items() if v is not None
    }

    with httpx.Client(timeout=10.0) as client:
        # Register once
        try:
            print(f"model name is : {MODEL_NAME}")
            print(f"session id is : {SESSION_ID}")
            client.post(
                f"{REGISTRY_URL}/register",
                json={
                    "name": MODEL_NAME,
                    "endpoint": f"{public_url}/transcribe_audio",
                    "session_id": SESSION_ID,
                },
                headers=headers
            )
            print("✓ Registered with registry")
        except Exception as e:
            print(f"Register failed: {e}")

        # Keep pinging
        while True:
            try:
                client.post(
                    f"{REGISTRY_URL}/ping",
                    json={
                        "name": MODEL_NAME,
                        "session_id": SESSION_ID,
                    },
                    headers=headers
                )
            except Exception as e:
                print(f"Ping failed: {e}")

            time.sleep(30)


# ── Main ──────────────────────────────────────────────────────────────────────
async def main():
    # Start ngrok ONCE
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

    print(f"ngrok URL: {public_url}")

    # Start registry thread ONCE
    threading.Thread(
        target=registry_loop,
        args=(public_url,),
        daemon=True
    ).start()

    # Start server
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()


# Run
await main()

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

ngrok URL: https://sparing-inbred-lina.ngrok-free.dev
model name is : seamless-large-M4T
session id is : 78240940-eb5b-4b8e-8b53-0f4aef4f9e3e
✓ Registered with registry


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     3.212.25.85:0 - "POST /transcribe_audio HTTP/1.1" 200 OK
INFO:     3.212.25.85:0 - "POST /transcribe_audio HTTP/1.1" 200 OK
INFO:     3.212.25.85:0 - "POST /transcribe_audio HTTP/1.1" 200 OK
